# Detroit Investment Properties — Your Picks

---

In [ ]:
import importlib, utils, utils.filters, utils.display, utils.loader
importlib.reload(utils.filters)
importlib.reload(utils.display)
importlib.reload(utils.loader)
importlib.reload(utils)

from utils import load_data, clean, investment_filter, flip_score
from utils.display import picks_table, quick_stats

In [ ]:
df_raw = load_data()

In [ ]:
df = flip_score(clean(df_raw))
df_inv = flip_score(investment_filter(df, min_beds=3, min_sqft=1000, price_min=60_000, price_max=150_000))
gold = df_inv[df_inv["cluster"] == "GOLD"].copy()

print(f"{len(df)} total listings  →  {len(df_inv)} match criteria  →  {len(gold)} GOLD picks")

## What We're Looking For

| Criteria | Value |
|----------|-------|
| Bedrooms | 3+ |
| Living area | 1,000+ sqft |
| Price | $60K – $150K |
| Best candidate | Cheaper per sqft than similar homes, sitting on market, seller motivated |

## Market Snapshot

In [ ]:
print(quick_stats(df))

## Your Picks (ranked by flip score)

**GOLD** = look at these first. **PASS** = no real upside. **AVOID** = too small to flip.

In [ ]:
from IPython.display import display
display(picks_table(df_inv))

## Why These? — Score Breakdown

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

target = gold if not gold.empty else df_inv.head(8)
if target.empty:
    print("No data.")
else:
    DARK = {"figure.facecolor": "#0e1117", "axes.facecolor": "#161b22",
            "axes.edgecolor": "#30363d", "axes.labelcolor": "#c9d1d9",
            "text.color": "#c9d1d9", "xtick.color": "#8b949e", "ytick.color": "#8b949e"}

    axes_cols = ["ax_value_gap", "ax_distress", "ax_market_reject", "ax_seller_weak"]
    axes_labels = ["Underpriced", "Distress", "Ignored by Market", "Seller Desperate"]
    colors = ["#58a6ff", "#f85149", "#d29922", "#bc8cff"]

    addrs = target["address"].fillna("?").str[:25].tolist()
    n = len(addrs)

    with plt.rc_context(DARK):
        fig, ax = plt.subplots(figsize=(10, max(3, n * 0.55)))
        y = np.arange(n)
        left = np.zeros(n)
        for col, label, color in zip(axes_cols, axes_labels, colors):
            vals = target[col].fillna(0).values
            ax.barh(y, vals, left=left, color=color, edgecolor="#0e1117", label=label, height=0.55)
            left += vals
        for i, total in enumerate(target["flip_score"].values):
            ax.text(left[i] + 0.15, i, str(int(total)), va="center", color="white", fontweight="bold")
        ax.set_yticks(y)
        ax.set_yticklabels(addrs, fontsize=9, color="#c9d1d9")
        ax.invert_yaxis()
        ax.set_xlabel("Flip Score (0–10)")
        ax.set_title("Why Each Property Scored High", fontsize=14, fontweight="bold", color="white")
        ax.legend(loc="lower right", fontsize=8, facecolor="#161b22", edgecolor="#30363d", labelcolor="#c9d1d9")
        ax.set_xlim(0, 11)
        ax.spines[:].set_color("#30363d")
        plt.tight_layout()
        plt.show()

## Save

In [ ]:
import os
os.makedirs("data", exist_ok=True)
df_inv.to_pickle("data/df_picks.pkl")
print(f"Saved → data/df_picks.pkl ({len(df_inv)} rows)")